# Behavioral Analysis
Compare success rate (_hit rate_) and sensitivity (_d'_ or _A'_) across different experimental conditions.

In [1]:
import polars as pl
import pandas as pd
from pymer4.models import glmer
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

import config as cnfg
import analysis.helpers.funnels.funnel_config as fcnfg
from analysis.helpers.read_data import read_data, parse_as_categorical
from analysis.helpers.funnels.trial_inclusion import check_trial_inclusion_criteria
from analysis.helpers.sdt import calc_sdt_metrics
from data_models.LWSEnums import SearchArrayCategoryEnum, ImageCategoryEnum

pio.renderers.default = "browser"      # or "browser"

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


## Prepare Data
### Read Data

In [2]:
loaded_data = read_data(cnfg.OUTPUT_PATH)
targets = loaded_data.targets
actions = loaded_data.actions
metadata = loaded_data.metadata
idents = loaded_data.identifications
fixations = loaded_data.fixations
del loaded_data

### Merge Identifications with Trial & Target Metadata

In [3]:
idents = (
    idents
    .merge(
        metadata[["subject", "trial", "trial_category"]], on=["subject", "trial"], how="left"
    )
    .merge(
        targets[["subject", "trial", "target", "category", "angle"]],
        on=["subject", "trial", "target"],
        how="left",
    )
    .rename(columns={"category": "target_category", "angle": "target_angle"})
    .astype({"subject": "category", "trial": int, "target": "category", "target_angle": float,})
    .assign(
        is_hit=lambda df: (df["identification_category"] == "hit").astype(bool),
        abs_target_angle=lambda df: df["target_angle"].abs().astype(float),
        trial_category=lambda df: parse_as_categorical(
            df["trial_category"], SearchArrayCategoryEnum, True
        ),
        target_category=lambda df: parse_as_categorical(
            df["target_category"], ImageCategoryEnum, False
        ),
    )
    .sort_values(["subject", "trial", "target"])
    .reset_index(drop=True)
)

### Calculate SDT Metric per Trial

In [4]:
trial_sdt_scores = calc_sdt_metrics(metadata, idents, "loglinear")

### Exclude Bad Trials

In [5]:
inclusion_criteria = check_trial_inclusion_criteria(
    metadata, fixations, actions, idents,
    min_gaze_coverage=fcnfg.DEFAULT_GAZE_COVERAGE_PERCENT_THRESHOLD,
    min_fixation_rate=fcnfg.DEFAULT_FIXATION_RATE_THRESHOLD,
    bad_actions=fcnfg.DEFAULT_BAD_ACTIONS,
    require_actions=False,
)

In [6]:
idents = idents.merge(
    inclusion_criteria["is_valid_trial"].reset_index(), on=["subject", "trial"], how="left"
)
idents = idents.loc[idents["is_valid_trial"]]      # apply trial filter

metadata = metadata.merge(
    inclusion_criteria["is_valid_trial"].reset_index(), on=["subject", "trial"], how="left"
)
metadata = metadata.loc[metadata["is_valid_trial"]]      # apply trial filter

trial_sdt_scores = trial_sdt_scores.merge(
    inclusion_criteria["is_valid_trial"].reset_index(), on=["subject", "trial"], how="left"
)
trial_sdt_scores = trial_sdt_scores.loc[trial_sdt_scores["is_valid_trial"]]

### Calculate SDT per Trial Category

In [7]:
scores_per_subject = (
    trial_sdt_scores
    .groupby("subject")
    .agg(
        n_trials=("trial_category", "count"),
        hit_rate=("hit_rate", "mean"),
        hit_rate_std=("hit_rate", "std"),
        fa_rate=("false_alarm_rate", "mean"),
        fa_rate_std=("false_alarm_rate", "std"),
        d_prime=("d_prime", "mean"),
        d_prime_std=("d_prime", "std"),
        a_prime=("a_prime", "mean"),
        a_prime_std=("a_prime", "std"),
        f1_score=("f1_score", "mean"),
        f1_score_std=("f1_score", "std"),
    )
    .assign(trial_category="all")
    .reset_index(drop=False)
)
scores_per_trial_category = (
    trial_sdt_scores
    .groupby(["subject", "trial_category"])
    .agg(
        n_trials=("trial_category", "count"),
        hit_rate=("hit_rate", "mean"),
        hit_rate_sem=("hit_rate", "sem"),
        fa_rate=("false_alarm_rate", "mean"),
        fa_rate_sem=("false_alarm_rate", "sem"),
        d_prime=("d_prime", "mean"),
        d_prime_sem=("d_prime", "sem"),
        a_prime=("a_prime", "mean"),
        a_prime_sem=("a_prime", "sem"),
        f1_score=("f1_score", "mean"),
        f1_score_sem=("f1_score", "sem"),
    )
    .reset_index(drop=False)
)
scores_per_trial_category = (
    pd.concat([scores_per_trial_category, scores_per_subject], ignore_index=True)
    .assign(
        trial_category=lambda df: parse_as_categorical(df["trial_category"], SearchArrayCategoryEnum, True),
    )
    .dropna(subset=["n_trials"])
    .sort_values(["subject", "trial_category"])
    .reset_index(drop=True)
)

## Descriptive Statistics
We calculate the percentage of _MISSED_ targets for each subject, and drill down to look at the statistics across trial types and target categories

In [95]:
def calc_miss_rate(groups):
    hits_and_misses = idents.loc[idents["identification_category"].isin(["hit", "miss"])]
    rates = (
        hits_and_misses
        .groupby(groups, observed=True)
        .agg(
            n_targets=("identification_category", "count"),
            n_misses=("identification_category", lambda x: (x == "miss").sum()),
        )
        .query("n_targets > 0")
        .fillna({"n_misses": 0})
        .assign(miss_rate=lambda df: df["n_misses"] / df["n_targets"],)
    )
    return rates


subjects_miss_rate = calc_miss_rate(["subject"])
average_miss_rate = subjects_miss_rate["miss_rate"].agg(["count", "mean", "sem"])
print(f"Average Miss-Rate across {average_miss_rate['count']} Subjects:\t{100 * average_miss_rate['mean'] :.2f} ± {100 * average_miss_rate['sem'] :.2f}%")

print(f"Average Miss-Rates across Trial Types:")
trialtype_miss_rate = (
    calc_miss_rate(["subject", "trial_category"])
    .groupby("trial_category", observed=True)["miss_rate"]
    .agg(["mean", "sem"])
)
display(trialtype_miss_rate)

print(f"Average Miss-Rates across Target Categories:")
targetcat_miss_rate = (
    calc_miss_rate(["subject", "target_category"])
    .groupby("target_category", observed=True)["miss_rate"]
    .agg(["mean", "sem"])
)
display(targetcat_miss_rate)

Average Miss-Rate across 12.0 Subjects:	24.44 ± 1.86%
Average Miss-Rates across Trial Types:


,mean,sem
trial_category,,
COLOR,0.208705,0.021778
BW,0.314761,0.034908
NOISE,0.213985,0.021786


Average Miss-Rates across Target Categories:


,mean,sem
target_category,,
ANIMAL_OTHER,0.319020,0.025200
HUMAN_OTHER,0.264265,0.028759
ANIMAL_FACE,0.411131,0.047843
HUMAN_FACE,0.158077,0.031198
OBJECT_HANDMADE,0.193833,0.026951
OBJECT_NATURAL,0.138351,0.028751


## Descriptive Figures
### SDT Measures by Trial Category

In [8]:
metrics = {"hit_rate": "Hit Rate", "f1_score": "$f1$ Score", "a_prime": "$A'$"}

fig = make_subplots(
    rows=len(metrics), cols=2, column_widths=[0.8, 0.2,],
    shared_yaxes=True, shared_xaxes=True, x_title="Trial Category",
    vertical_spacing=0.075, horizontal_spacing=0.025,
    subplot_titles=[ttl for tup in zip(metrics.values(), [""] * len(metrics)) for ttl in tup]
)
for r, (metric, metric_name) in enumerate(metrics.items()):
    for c, is_aggregated in enumerate([False, True]):
        if is_aggregated:
            data = scores_per_trial_category[scores_per_trial_category["trial_category"] == "all"]
        else:
            data = scores_per_trial_category[scores_per_trial_category["trial_category"] != "all"]
        # add distribution plots
        fig.add_trace(
            row=r+1, col=c+1, trace=go.Violin(
                x=data["trial_category"], y=data[metric],
                name=f"{metric_name} Distribution",
                marker=dict(color="gray", line=dict(color="black", width=1)),
                opacity=0.75, points=False, spanmode="hard", width=0.5,
                box=dict(visible=False), meanline=dict(visible=True),
                showlegend=False,
            ))
        # add individual subject points
        for i, subj in enumerate(data["subject"].unique()):
            subj_data = data[data["subject"] == subj]
            subj_name = f"Subject {subj}"
            subj_color = cnfg.get_discrete_color(i, loop=True)
            subj_color = (int(subj_color[1:3], 16), int(subj_color[3:5], 16), int(subj_color[5:7], 16))
            fig.add_trace(
                row=r+1, col=c+1, trace=go.Scatter(
                    x=subj_data["trial_category"], y=subj_data[metric],
                    error_y=dict(
                        type="data", array=subj_data[f"{metric}_std"],
                        color=f"rgba{subj_color + (1,)}", thickness=1.5, width=3,
                        visible=is_aggregated,
                    ),
                    name=subj_name, legendgroup=subj_name, showlegend=(r==0 and c==0),
                    mode="markers+lines",
                    marker=dict(size=6, color=f"rgba{subj_color + (1,)}"),
                    line=dict(width=2, color=f"rgba{subj_color + (0.5,)}"),
                    text=f"<b>Subject</b>:\t{subj}", hoverinfo="text+y",
            ))

fig.update_layout(
    width=1000, height=1000,
    title=dict(text="Behavioral Measures across Trial Types"),
)
fig.show()

### Hit-Rate per Target Category

In [9]:
grouped = []
for groupers in [["subject"], ["subject", "target_category"]]:
    hit_rate = (
        idents
        .loc[idents["identification_category"] != "false_alarm",]
        .groupby(groupers, observed=False)
        .agg(
            n_idents=("identification_category", "count"),
            n_hits=("identification_category", lambda x: (x == "hit").sum()),
        )
        .assign(hit_rate=lambda df: df["n_hits"] / df["n_idents"],)
        .dropna(subset=["hit_rate"])
        .reset_index(drop=False)
    )
    if "target_category" not in groupers:
        hit_rate["target_category"]  = "all"
    grouped.append(hit_rate)

hit_rate_stats = (
    pd.concat(grouped)
    .assign(
        target_category=lambda df: parse_as_categorical(df["target_category"], ImageCategoryEnum, False)
    )
    .sort_values(["subject", "target_category"])
    .reset_index(drop=True)
)

In [10]:
fig = make_subplots(
    rows=1, cols=2, column_widths=[0.8, 0.2,],
    shared_yaxes=True, shared_xaxes=True,
    subplot_titles=["Hit Rate", ""], x_title="Target Category",
    vertical_spacing=0.1, horizontal_spacing=0.05,
)
for c, is_aggregated in enumerate([False, True]):
    if is_aggregated:
        data = hit_rate_stats[hit_rate_stats["target_category"] == "all"]
    else:
        data = hit_rate_stats[hit_rate_stats["target_category"] != "all"]
    # add distribution plots
    fig.add_trace(
        row=1, col=c+1, trace=go.Violin(
            x=data["target_category"], y=data["hit_rate"],
            name="Hit Rate Distribution",
            marker=dict(color="gray", line=dict(color="black", width=1)),
            opacity=0.75, points=False, spanmode="hard", width=0.75,
            box=dict(visible=False), meanline=dict(visible=True),
            showlegend=False,
        ))
    # add individual subject points
    for i, subj in enumerate(data["subject"].unique()):
        subj_data = data[data["subject"] == subj]
        subj_name = f"Subject {subj}"
        subj_color = cnfg.get_discrete_color(i, loop=True)
        subj_color = (int(subj_color[1:3], 16), int(subj_color[3:5], 16), int(subj_color[5:7], 16))
        fig.add_trace(
            row=1, col=c+1, trace=go.Scatter(
                x=subj_data["target_category"], y=subj_data["hit_rate"],
                name=subj_name, legendgroup=subj_name, showlegend=(c==0),
                mode="markers+lines",
                marker=dict(size=6, color=f"rgba{subj_color + (1,)}"),
                line=dict(width=2, color=f"rgba{subj_color + (0.25,)}"),
                text=f"<b>Subject</b>:\t{subj}", hoverinfo="text+y",
        ))

fig.update_xaxes(
    row=1, col=1,
    tickmode="array",
    tickvals=hit_rate_stats["target_category"].unique(),
    ticktext=[cat.replace("_", "<br>") for cat in hit_rate_stats["target_category"].unique()]
)
for ann in fig.layout.annotations:
    if ann.text == "Target Category":
        ann.x=0.375
        ann.xanchor="center"
        ann.y = -0.01
fig.update_layout(
    width=900, height=600,
    title=dict(text="Hit Rate by Target Category"),
)
fig.show()

### Hit-Rate by (Absolute) Target Angle

In [11]:
trial_symbols = {
    "all": "circle", 'BW': 'x', 'COLOR': 'diamond', "NOISE": "square",
}
target_symbols = {
    'OBJECT_HANDMADE': 'cross', 'OBJECT_NATURAL': 'x',
    'ANIMAL_OTHER': 'square', 'ANIMAL_FACE': 'diamond',
    'HUMAN_OTHER': 'star', 'HUMAN_FACE': 'hexagram',
}

grouped = []
for groupers in [["subject", "abs_target_angle"], ["subject", "abs_target_angle", "trial_category"]]:
    hit_rate = (
        idents
        .loc[idents["identification_category"] != "false_alarm",]
        .groupby(groupers, observed=False)
        .agg(
            n_idents=("identification_category", "count"),
            n_hits=("identification_category", lambda x: (x == "hit").sum()),
        )
        .assign(hit_rate=lambda df: df["n_hits"] / df["n_idents"],)
        .dropna(subset=["hit_rate"])
        .reset_index(drop=False)
    )
    if "trial_category" not in groupers:
        hit_rate["trial_category"]  = "all"
    grouped.append(hit_rate)

hit_rate_over_subjects = (
    pd.concat(grouped)
    .assign(
        trial_category=lambda df: parse_as_categorical(df["trial_category"], SearchArrayCategoryEnum, False)
    )
    .groupby(["abs_target_angle", "trial_category"], observed=False)   # aggregate over subjects
    .agg(
        N=("hit_rate", "count"),
        average_rate=("hit_rate", "mean"),
        sem_rate=("hit_rate", "sem"),
    )
    .sort_index()
    .reset_index(drop=False)
    .assign(
        color=lambda df: df['trial_category'].map(lambda cat: cnfg.get_discrete_color(
                df['trial_category'].unique().tolist().index(cat), loop=True
        )),
        symbol=lambda df: df['trial_category'].map(trial_symbols)
    )
)

In [12]:
fig = go.Figure()
for category in hit_rate_over_subjects["trial_category"].unique():
    category_data = hit_rate_over_subjects[hit_rate_over_subjects["trial_category"] == category]
    fig.add_trace(go.Scatter(
        x=category_data["abs_target_angle"],
        y=category_data["average_rate"],
        error_y=dict(
            type="data",
            array=category_data["sem_rate"],
            visible=True,
        ),
        mode="markers",
        name=category,
        marker=dict(size=8, symbol=category_data["symbol"], color=category_data["color"],),
        line=dict(color=category_data["color"].iloc[0], width=2),
    ))

fig.update_layout(
    title=dict(
        text="Hit Rate by Absolute Target Angle<br>(averaged across subjects)",
        font=cnfg.TITLE_FONT
    ),
    xaxis=dict(title=dict(text="Absolute Target Rotation Angle (﷿﷿)")),
    yaxis=dict(title=dict(text="Hit Rate"), range=[-0.05, 1.05]),
)
fig.show()

## Bayesian Hierarchical Analysis
### Hit-Rate against Icon Features
We fit (using `pymer4` and _R_'s `lme` packages) the hierarchical Generalized Linear Model (hGLM):
$$ \text{logit}(p[hit]) \sim abs(angle) * C(trial\_category) * C(target\_category) + (1 | subject) $$

In [13]:
hits_for_angle_regression = idents.loc[idents["identification_category"] != "false_alarm",]
model = glmer(
    formula="is_hit ~ abs_target_angle * trial_category * target_category + (1 | subject)",
    data=pl.from_pandas(hits_for_angle_regression),
    family="binomial",      # logistic regression in R uses `Binomial` (with parameter `n=1`)
)
model.set_factors(["subject", "trial_category", "target_category"])
result = model.fit(exponentiate=True, summary=True)
print(model.convergence_status)

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.003565234

R messages: 
Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.003565234

Convergence status
: [1] FALSE
attr(,"gradient")
[1] 0.003565234



In [14]:
model.anova()
model.summary_anova()

GT(_tbl_data=shape: (7, 7)
┌─────────────────────────────────┬──────┬─────┬─────────┬────────┬─────────┬───────┐
│ model term                      ┆ df1  ┆ df2 ┆ F_ratio ┆ Chisq  ┆ p_value ┆ stars │
│ ---                             ┆ ---  ┆ --- ┆ ---     ┆ ---    ┆ ---     ┆ ---   │
│ str                             ┆ f64  ┆ f64 ┆ f64     ┆ f64    ┆ str     ┆ str   │
╞═════════════════════════════════╪══════╪═════╪═════════╪════════╪═════════╪═══════╡
│ abs_target_angle                ┆ 1.0  ┆ inf ┆ 0.01    ┆ 0.01   ┆ 0.9187  ┆       │
│ trial_category                  ┆ 2.0  ┆ inf ┆ 13.407  ┆ 26.814 ┆ <.001   ┆ ***   │
│ target_category                 ┆ 5.0  ┆ inf ┆ 12.542  ┆ 62.71  ┆ <.001   ┆ ***   │
│ abs_target_angle:trial_categor… ┆ 2.0  ┆ inf ┆ 1.766   ┆ 3.532  ┆ 0.1711  ┆       │
│ abs_target_angle:target_catego… ┆ 5.0  ┆ inf ┆ 0.519   ┆ 2.595  ┆ 0.7619  ┆       │
│ trial_category:target_category  ┆ 10.0 ┆ inf ┆ 1.579   ┆ 15.79  ┆ 0.1059  ┆       │
│ abs_target_angle:trial_categor… ┆ 10.0 ┆ inf ┆ 1.21    ┆ 12.1   ┆ 0.2783  ┆       │
└─────────────────────────────────┴──────┴─────┴─────────┴────────┴─────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x000001D0ACECE6C0>, _boxhead=Boxhead([ColInfo(var='model term', type=<ColInfoTypeEnum.default: 1>, column_label='model term', column_align='left', column_width=None), ColInfo(var='df1', type=<ColInfoTypeEnum.default: 1>, column_label='df1', column_align='right', column_width=None), ColInfo(var='df2', type=<ColInfoTypeEnum.default: 1>, column_label='df2', column_align='right', column_width=None), ColInfo(var='F_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='F_ratio', column_align='right', column_width=None), ColInfo(var='Chisq', type=<ColInfoTypeEnum.default: 1>, column_label='Chisq', column_align='right', column_width=None), ColInfo(var='p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p_value', column_align='left', column_width=None), ColInfo(var='stars', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001D0ACECBF80>, _spanners=Spanners([]), _heading=Heading(title='ANOVA (Type III tests)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001D0ACED2E10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001D0ACED0EF0>, _source_notes=[Md(text='Signif. codes: *0 *** 0.001 ** 0.01 * 0.05 . 0.1*')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001D0ACED0D40>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001D0ACED0890>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_c

#### Marginal Effect: Trial Categories

In [15]:
model.emmeans("trial_category")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



trial_category,prob,SE,df,asymp_LCL,asymp_UCL
cat,f64,f64,f64,f64,f64
"""BW""",0.665863,0.028236,inf,0.595441,0.729594
"""COLOR""",0.807038,0.022898,inf,0.746453,0.85594
"""NOISE""",0.80816,0.022707,inf,0.748077,0.856658


In [16]:
model.emmeans("trial_category", contrasts="pairwise", p_adjust="tukey")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
cat,f64,f64,f64,f64,f64,f64,f64,f64
"""BW / COLOR""",0.476472,0.08176,inf,0.318698,0.712354,1.0,-4.320338,0.000046
"""BW / NOISE""",0.473046,0.081242,inf,0.316296,0.707477,1.0,-4.358629,0.000039
"""COLOR / NOISE""",0.992808,0.185984,inf,0.640014,1.540073,1.0,-0.038528,0.999182


#### Margeinal Effect: Target Categories

In [17]:
model.emmeans("target_category")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



target_category,prob,SE,df,asymp_LCL,asymp_UCL
cat,f64,f64,f64,f64,f64
"""ANIMAL_FACE""",0.589925,0.037988,inf,0.487629,0.684992
"""ANIMAL_OTHER""",0.65226,0.03682,inf,0.550301,0.741942
"""HUMAN_FACE""",0.847056,0.027525,inf,0.760005,0.90642
"""HUMAN_OTHER""",0.733272,0.031923,inf,0.641495,0.808565
"""OBJECT_HANDMADE""",0.805421,0.029583,inf,0.715831,0.871823
"""OBJECT_NATURAL""",0.878773,0.02472,inf,0.797435,0.930305


In [18]:
model.emmeans("target_category", contrasts="pairwise", p_adjust="tukey")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
cat,f64,f64,f64,f64,f64,f64,f64,f64
"""ANIMAL_FACE / ANIMAL_OTHER""",0.766953,0.159194,inf,0.424503,1.385658,1.0,-1.278286,0.797052
"""ANIMAL_FACE / HUMAN_FACE""",0.259749,0.06462,inf,0.127839,0.527768,1.0,-5.41867,8.9608e-7
"""ANIMAL_FACE / HUMAN_OTHER""",0.523285,0.108878,inf,0.289222,0.946769,1.0,-3.1126,0.022837
"""ANIMAL_FACE / OBJECT_HANDMADE""",0.347542,0.079544,inf,0.181029,0.667218,1.0,-4.617637,0.000057
"""ANIMAL_FACE / OBJECT_NATURAL""",0.198453,0.052731,inf,0.093071,0.423157,1.0,-6.086376,1.7302e-8
…,…,…,…,…,…,…,…,…
"""HUMAN_FACE / OBJECT_HANDMADE""",1.337992,0.360338,inf,0.621083,2.88242,1.0,1.081162,0.88907
"""HUMAN_FACE / OBJECT_NATURAL""",0.764018,0.230191,inf,0.32376,1.802951,1.0,-0.893371,0.948224
"""HUMAN_OTHER / OBJECT_HANDMADE""",0.664155,0.154249,inf,0.342639,1.287366,1.0,-1.762082,0.490517
